In [1]:
# S&P 500 Sector Performance Analysis
# This notebook walks through the full analysis pipeline
# pulling live market data, calculating returns, and visualizing sector performance

In [2]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import date

SECTORS = {
    "Technology":        "XLK",
    "Healthcare":        "XLV",
    "Financials":        "XLF",
    "Consumer Discret.": "XLY",
    "Consumer Staples":  "XLP",
    "Industrials":       "XLI",
    "Energy":            "XLE",
    "Utilities":         "XLU",
    "Real Estate":       "XLRE",
    "Materials":         "XLB",
    "Communication":     "XLC",
}

tickers = list(SECTORS.values())
today = date.today()
pull_start = date(today.year - 1, 1, 1)

prices = yf.download(tickers, start=pull_start, end=today, auto_adjust=True, progress=False)["Close"]
prices.tail()

Ticker,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,
2026-03-11,49.840000,117.000000,56.980000,49.639999,169.490005,140.429993,84.589996,42.410000,46.169998,152.850006,114.139999
2026-03-12,49.680000,115.269997,57.509998,48.830002,165.240005,137.839996,84.250000,42.139999,46.500000,150.160004,111.519997
2026-03-13,49.189999,114.449997,57.700001,48.889999,164.649994,136.800003,84.739998,42.250000,46.959999,149.789993,110.860001
2026-03-16,49.400002,115.330002,57.900002,49.299999,166.059998,138.779999,84.980003,42.580002,47.259998,151.009995,112.199997
2026-03-17,49.520000,115.370003,58.509998,49.560001,166.500000,139.539993,84.699997,42.720001,47.130001,149.639999,113.180000


In [3]:
def period_return(df, start):
    start_ts = pd.Timestamp(start)
    valid = df.index[df.index >= start_ts]
    if len(valid) == 0:
        return pd.Series([None] * len(df.columns), index=df.columns)
    start_price = df.loc[valid[0]]
    end_price = df.iloc[-1]
    return ((end_price - start_price) / start_price * 100).round(2)

ytd_start = date(today.year, 1, 1)
three_m = pd.Timestamp.today() - pd.DateOffset(months=3)
one_m = pd.Timestamp.today() - pd.DateOffset(months=1)

r1m = period_return(prices, one_m)
r3m = period_return(prices, three_m)
rytd = period_return(prices, ytd_start)

summary = pd.DataFrame({
    "Sector": list(SECTORS.keys()),
    "Ticker": tickers,
    "1M (%)": [r1m[t] for t in tickers],
    "3M (%)": [r3m[t] for t in tickers],
    "YTD (%)": [rytd[t] for t in tickers],
}).sort_values("YTD (%)", ascending=False).reset_index(drop=True)

summary.insert(0, "Rank", range(1, len(summary) + 1))
summary

,Rank,Sector,Ticker,1M (%),3M (%),YTD (%)
0,1,Energy,XLE,6.03,33.72,28.17
1,2,Utilities,XLU,2.21,11.38,9.15
2,3,Consumer Staples,XLP,-3.39,8.38,9.02
3,4,Materials,XLB,-6.27,9.71,7.37
4,5,Real Estate,XLRE,-1.11,7.08,5.79
5,6,Industrials,XLI,-5.58,7.19,5.39
6,7,Communication,XLC,0.20,-0.67,-1.31
7,8,Technology,XLK,-0.48,-3.35,-3.30
8,9,Healthcare,XLV,-4.85,-3.01,-3.77
9,10,Consumer Discret.,XLY,-2.63,-6.93,-4.37


In [4]:
print("Top 3 YTD performers:")
print(summary.head(3)[['Sector', 'YTD (%)']].to_string(index=False))
print("\nBottom 3 YTD performers:")
print(summary.tail(3)[['Sector', 'YTD (%)']].to_string(index=False))

Top 3 YTD performers:
          Sector  YTD (%)
          Energy    28.17
       Utilities     9.15
Consumer Staples     9.02

Bottom 3 YTD performers:
           Sector  YTD (%)
       Healthcare    -3.77
Consumer Discret.    -4.37
       Financials    -9.78
